# Level 2: Simplicial Complexes, Cochains, and the L1 Norm

**Paper Package 002** — Pedagogical Progression: Level 2 of 5

**Target Audience:** Physics, Mathematics, or Computer Science undergraduates with working knowledge of linear algebra and vector calculus.

**Core Idea:** Physical laws on discrete lattices depend critically on the choice of norm. The Euclidean $\ell^2$ norm distributes errors smoothly everywhere, while the $\ell^1$ norm preserves sharp topological boundaries and enforces sparse, localized defect repairs.

*Evidence scope: finite toy-model demonstration only.*

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Locate and import code directory dynamically
cwd = Path.cwd().resolve()
for candidate in (cwd, cwd / "code", cwd.parent / "code", *(d / "code" for d in cwd.parents)):
    if (candidate / "constraint_closure_l1_structure.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from poset import Poset
from growth_iterator import CellGrowthIterator
print('[OK] Level 2 environment ready!')

## 1. Simplicial Complex and the Chain Complex Property $d_1 d_0 = 0$

In a discrete simplicial complex:
- $C^0$: 0-cochains on vertices $V$.
- $C^1$: 1-cochains on directed edges $E$.
- $C^2$: 2-cochains on oriented triangular faces $F$.

The discrete exterior derivatives are:
$$C^0 \xrightarrow{d_0} C^1 \xrightarrow{d_1} C^2.$$

The fundamental identity of algebraic topology is **nilpotency**:
$$d_1 \circ d_0 = 0,$$
which states that the "curl of a gradient" vanishes identically on every cell.

Let's construct an oriented 2-simplex (a filled triangle with vertices 0, 1, 2 and face $(0, 1, 2)$):

In [ ]:
# Edges: e0 = (0, 1), e1 = (1, 2), e2 = (0, 2)
# Face: f0 = (0, 1, 2) whose boundary is e0 + e1 - e2
d0 = np.array([
    [-1.0,  1.0,  0.0],  # e0: 0 -> 1
    [ 0.0, -1.0,  1.0],  # e1: 1 -> 2
    [-1.0,  0.0,  1.0],  # e2: 0 -> 2
])

d1 = np.array([
    [1.0, 1.0, -1.0]     # f0: e0 + e1 - e2
])

# Verify d1 @ d0 == 0
composition = d1 @ d0
print(f"d1 @ d0 matrix:\n{composition}")
assert np.allclose(composition, 0.0)
print("[OK] Nilpotency d1 . d0 = 0 verified analytically!")

## 2. The Midpoint Subdivision Fallacy in $\ell^1$

A common intuition from numerical PDE methods is that subdividing a mesh reduces errors.
However, in pure $\ell^1$ total variation, ordinary midpoint subdivision **preserves total variation exactly**:

Let an edge have vertex values $a$ and $b$. Its raw difference is $|a - b|$.
If we insert a new midpoint vertex $w$ with linear value $s(w) = \frac{a+b}{2}$, the two sub-edges have differences:
$$\left|a - \frac{a+b}{2}\right| + \left|\frac{a+b}{2} - b\right| = \left|\frac{a-b}{2}\right| + \left|\frac{a-b}{2}\right| = |a - b|.$$

The total variation is unchanged! Let's test this in Python:

In [ ]:
a, b = 3.0, 7.0
orig_variation = abs(b - a)

midpoint = (a + b) / 2.0
subdivided_variation = abs(midpoint - a) + abs(b - midpoint)

print(f"Original edge variation:   {orig_variation}")
print(f"Subdivided edge variation: {subdivided_variation}")

assert np.isclose(orig_variation, subdivided_variation)
print("[OK] Verified: Linear midpoint subdivision preserves L1 total variation.")

## 3. Comparing $\ell^1$ vs $\ell^2$ Gauge Repairs

Given a 1-cochain $\omega$ on a network, we seek the closest exact cochain $d_0 \alpha$:
- Under the $\ell^2$ norm, the projection $\alpha_{\ell^2} = (d_0^T d_0)^+ d_0^T \omega$ minimizes $\|\omega - d_0 \alpha\|_2^2$. This is the standard Euclidean Hodge projection.
- Under the $\ell^1$ norm, the quotient residual $\Phi_1([\omega]) = \min_\alpha \|\omega - d_0 \alpha\|_1$ is solved by Linear Programming.

Let's compare them on the 3-cycle with cochain $\omega = (0, 1, 0)$:

In [ ]:
from l1_topological_defects import evaluate_c3_hodge_split_inflation

omega = [0.0, 1.0, 0.0]
split_res = evaluate_c3_hodge_split_inflation(omega)

print(f"Cochain omega:              {omega}")
print(f"L2 exact component d0*alpha:{split_res['d0_alpha_l2']}")
print(f"L2 harmonic component h:    {split_res['h_l2_harmonic']}")
print(f"L1 quotient residual Phi_1: {split_res['phi_1']:.6f}")
print(f"L2-induced split ratio:     {split_res['r_split']:.6f}")

# For C3, the worst-case inflation ratio between L2 Hodge representative
# and true L1 quotient is exactly R_split = 7/3 = 2.333333!
assert np.isclose(split_res["r_split"], 7.0 / 3.0)
print("[OK] Hodge split inflation ceiling 7/3 verified on C3!")

## 4. Key Takeaways from Level 2

1. **Nilpotency:** $d_1 d_0 = 0$ ensures exact cochains have zero curl.
2. **Subdivision Invariance:** Midpoint subdivision does not reduce $\ell^1$ total variation; any modeled reduction in an LP must agree with the realized geometric operation.
3. **Norm Distortion:** The Euclidean Hodge decomposition spreads circulation across all edges ($h = (1/3, 1/3, 1/3)$), inflating the sum of norms to $7/3$, whereas the true $\ell^1$ quotient norm is $1.0$.

In **Level 3**, we will prove the general cycle inflation theorem $R_{\mathrm{split}}^{\max}(C_n) = 3 - 2/n$ and build the exact Primal-Dual Linear Program for $\Phi_1$.